# LASSO Regresyonu (L1 Regularization)

## Temel Kavramlar

Regresyon probleminde:

- `n` → gözlem sayısı (satır sayısı)
- `p` → özellik sayısı (sütun sayısı)
- `y` → tahmin edilmek istenen hedef değer
- `β` (beta) → modelin öğrendiği katsayılar
- `β̂` (beta şapka) → tahmin edilen katsayı, şapka sembolü "tahmin edilmiş" anlamına gelir

---

## OLS ve Ridge

OLS hata karesini minimize eder, katsayı büyüklüğünü umursamaz:

$$\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Ridge buna L2 cezası ekler — katsayıları küçültür ama sıfır yapmaz:

$$\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} \beta_j^2$$

---

## LASSO 

LASSO (Least Absolute Shrinkage and Selection Operator), Ridge'den farklı olarak
katsayıların **mutlak değerlerinin toplamını** cezalandırır — buna **L1 cezası** denir:

$$\min_{\beta} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} |\beta_j|$$

Sembollerin anlamı:
- $|\beta_j|$ → $j$. katsayının mutlak değeri (işaretinden bağımsız büyüklüğü)
- $\lambda$ → cezanın şiddeti, büyüdükçe daha fazla katsayı sıfıra çekilir
- $i$ → gözlem indisi (1'den n'e)
- $j$ → özellik indisi (1'den p'ye)

Matris formunda:

$$\min_{\beta} \| y - X\beta \|^2 + \lambda \|\beta\|_1$$

Burada $\|\beta\|_1 = \sum_{j=1}^{p} |\beta_j|$ — **L1 normu**, katsayıların mutlak değerlerinin toplamı.

---

## Ridge ile Temel Fark

| | Ridge (L2) | LASSO (L1) |
|---|---|---|
| Ceza terimi | $\sum \beta_j^2$ | $\sum |\beta_j|$ |
| Katsayı sıfır olur mu? | Hayır | Evet |
| Özellik seçimi yapar mı? | Hayır | Evet |
| Kapalı form çözümü var mı? | Evet | Hayır |
| Ne zaman tercih edilir? | Tüm özellikler önemliyse | Çok sayıda gereksiz özellik varsa |

#

## Neden LASSO Katsayıları Sıfır Yapabilir, Ridge Yapamaz?

Geometrik yorumla açıklanır.

<div style="display: flex; gap: 20px;">
  <img src="../figures/geo-lasso.png" width="45%"/>
</div>

Grafiği okuma rehberi:
- Elips şeklindeki çizgiler = hata yüzeyi (merkeze yaklaştıkça hata azalır, OLS çözümü merkezde)
- Mavi elmas = L1 kısıtı (LASSO): $|\beta_1| + |\beta_2| \leq t$
- Kırmızı daire = L2 kısıtı (Ridge): $\beta_1^2 + \beta_2^2 \leq t$
- Siyah nokta = OLS çözümü (kısıtsız en iyi nokta)
- Mavi yıldız = LASSO çözümü
- Kırmızı yıldız = Ridge çözümü

Hata elipsleri dışarıdan daralarak kısıt bölgesine ilk dokunduğu noktada durur.

**LASSO:** Elmas şeklinin köşeleri eksenlerin üzerindedir.
Elipsler büyük ihtimalle bu köşeye çarpar → bir katsayı tam sıfır olur.

**Ridge:** Dairenin her noktası eğridir, köşe yoktur.
Elipsler eksenden uzak bir noktaya çarpar → hiçbir katsayı tam sıfır olmaz.

Bu geometrik fark, LASSO'nun özellik seçimi yapabilmesinin matematiksel temelidir.

##

## Neden Kapalı Form Çözümü Yok?

Ridge'de türev alıp sıfıra eşitleyince doğrudan matris formülü çıkıyordu:

$$\hat{\beta}_{Ridge} = (X^T X + \lambda I)^{-1} X^T y$$

LASSO'da ceza terimi $|\beta_j|$ kullanıyor. Mutlak değer fonksiyonu $\beta_j = 0$
noktasında **türevlenemez** — orada bir "kırılma" (kink) vardır:

$$|\beta_j| = \begin{cases} \beta_j & \beta_j > 0 \\ -\beta_j & \beta_j < 0 \end{cases}$$

$\beta_j = 0$ noktasında türev tanımsızdır. Bu yüzden "türevi sıfıra eşitle" yaklaşımı
doğrudan uygulanamaz. Bunun yerine **subgradient** (altgradyan) kavramı kullanılır.

### Subgradient Nedir?

Türevlenebilir noktalarda türev tektir. $\beta_j = 0$ noktasında ise $|\beta_j|$'nin
türevi tanımsız olduğundan, bu noktada $[-1, 1]$ aralığındaki herhangi bir değer
"subgradient" olarak kabul edilir.

$$\partial|\beta_j| = \begin{cases} \{+1\} & \beta_j > 0 \\ [-1, +1] & \beta_j = 0 \\ \{-1\} & \beta_j < 0 \end{cases}$$

Bu subgradient yapısı, LASSO'nun bazı katsayıları **tam olarak sıfıra** çekebilmesini
sağlar. Eğer verinin katsayıyı sıfırdan uzaklaştırma gücü $\lambda$'yı aşmıyorsa,
optimal çözüm $\beta_j = 0$ olur.

Özetle: türevlenemeyen nokta bir engel gibi görünür ama aslında LASSO'nun özellik
seçimi yapabilmesinin matematiksel kaynağıdır.

### Çözüm Nasıl Bulunur?

Kapalı form olmadığı için iteratif (döngüsel) optimizasyon algoritmaları kullanılır.
En yaygın iki yöntem:

- **Coordinate Descent:** Her seferinde bir katsayıyı optimize eder, diğerlerini sabit tutar.
  Scikit-learn'ün varsayılan yöntemi budur.
- **ISTA (Iterative Shrinkage-Thresholding Algorithm):** Gradient adımı + soft-thresholding
  adımını dönüşümlü uygular.

#

## Soft-Thresholding: LASSO'nun Kalbi

Coordinate descent ile her katsayı için şu kural uygulanır:

$$\hat{\beta}_j = S\left(\hat{\beta}_j^{OLS}, \lambda\right)$$

Burada $S$ **soft-thresholding operatörü**dür:

$$S(z, \lambda) = \begin{cases} z - \lambda & z > \lambda \\ 0 & |z| \leq \lambda \\ z + \lambda & z < -\lambda \end{cases}$$

Sezgisel açıklama:
- Katsayı $\lambda$'dan büyükse: $\lambda$ kadar küçült
- Katsayı $-\lambda$'dan küçükse: $\lambda$ kadar büyüt (sıfıra doğru)
- Katsayı $[-\lambda, \lambda]$ arasındaysa: tam sıfır yap

Bu kural LASSO'nun özellik seçimini mümkün kılan mekanizmadır.
Ridge'de böyle bir eşik (threshold) yoktur — katsayılar hiçbir zaman tam sıfır olmaz.

## Regularization Path

<div style="display: flex; gap: 20px;">
  <img src="../figures/lasso-works.png" width="45%"/>
</div>

Grafiği okuma rehberi:
- X ekseni = $\lambda$ (regularization strength), logaritmik ölçek
- Y ekseni = katsayı değeri
- Her renkli çizgi = bir özelliğin katsayısı
- Gerçek katsayısı sıfır olan özellikler (Feature 4, 5) hemen sıfıra çekilir
- Önemli özellikler (Feature 1, 2, 3) daha uzun süre hayatta kalır

Bu Ridge'den temel farktır: Ridge'de hiçbir katsayı tam sıfır olmazken,
LASSO'da katsayılar adım adım tam sıfıra düşer.

## Soft-Thresholding Elle Hesaplama

3 özellikli bir modelde katsayılar şöyle olsun:

- $\beta_1 = 0.8$ (metrekare)
- $\beta_2 = -0.3$ (şehir merkezine uzaklık)
- $\beta_3 = 0.1$ (banyo sayısı)

### L1 Normu

$$\|\beta\|_1 = |0.8| + |-0.3| + |0.1| = 0.8 + 0.3 + 0.1 = 1.2$$

### $\lambda = 0$ (Ceza yok)

Ceza terimi: $0 \times 1.2 = 0$

LASSO, OLS'ye eşit hale gelir. Katsayılar değişmez:
$$\beta_1 = 0.8, \quad \beta_2 = -0.3, \quad \beta_3 = 0.1$$

### $\lambda = 0.5$ (Orta düzey ceza)

Soft-thresholding kuralı uygulanır:

- $\beta_1 = 0.8 > 0.5$ → $0.8 - 0.5 = 0.3$ (küçüldü, sıfır olmadı)
- $\beta_2 = -0.3$, $|\beta_2| = 0.3 < 0.5$ → **tam sıfır**
- $\beta_3 = 0.1$, $|\beta_3| = 0.1 < 0.5$ → **tam sıfır**

Sonuç: Model sadece $\beta_1$ ile çalışır. 2 özellik elendi.

### $\lambda = 2.0$ (Güçlü ceza)

- $\beta_1 = 0.8$, $|\beta_1| = 0.8 < 2.0$ → **tam sıfır**
- $\beta_2$, $\beta_3$ zaten sıfır

Sonuç: Tüm katsayılar sıfır, model sadece ortalama tahmin eder — underfitting.

---

| $\lambda$ | $\beta_1$ | $\beta_2$ | $\beta_3$ | Seçilen özellik sayısı |
|---|---|---|---|---|
| 0.0 | 0.80 | -0.30 | 0.10 | 3 |
| 0.5 | 0.30 | 0.00 | 0.00 | 1 |
| 2.0 | 0.00 | 0.00 | 0.00 | 0 |

#

In [1]:
import numpy as np

def soft_threshold(z, lam):
    """
    LASSO'nun temel operatörü.
    z  : katsayı değeri
    lam: lambda (regularization parametresi)
    """
    if z > lam:
        return z - lam
    elif z < -lam:
        return z + lam
    else:
        return 0.0


# --- Örnek: üç katsayı, farklı lambda değerleri ---
betas = [0.8, -0.3, 0.1]
lambdas = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]

print(f"{'Lambda':>8}  {'β1':>8}  {'β2':>8}  {'β3':>8}  {'Seçilen':>8}")
print("-" * 50)
for lam in lambdas:
    shrunk = [soft_threshold(b, lam) for b in betas]
    selected = sum(1 for b in shrunk if b != 0)
    print(f"{lam:>8.2f}  {shrunk[0]:>8.3f}  {shrunk[1]:>8.3f}  {shrunk[2]:>8.3f}  {selected:>8}")

  Lambda        β1        β2        β3   Seçilen
--------------------------------------------------
    0.00     0.800    -0.300     0.100         3
    0.05     0.750    -0.250     0.050         3
    0.10     0.700    -0.200     0.000         2
    0.20     0.600    -0.100     0.000         2
    0.50     0.300     0.000     0.000         1
    1.00     0.000     0.000     0.000         0
    2.00     0.000     0.000     0.000         0


## Lambda Seçimi: Cross-Validation

Lambda bir hiperparametredir — veriden değil, dışarıdan belirlenir.
Yanlış seçilirse:

- Çok küçük $\lambda$ → ceza zayıf → OLS gibi davranır, overfitting riski
- Çok büyük $\lambda$ → ceza güçlü → gereksiz özellikler de dahil sıfır olur, underfitting

Doğru seçim yöntemi: **cross-validation**

### K-Fold Cross-Validation Mantığı

1. Eğitim verisini K parçaya böl (genellikle K=5 veya K=10)
2. Her lambda değeri için:
   - K-1 parçayla eğit, kalan 1 parçayla test et
   - Bunu K kez tekrarla, her seferinde farklı parça test olsun
   - K test hatasının ortalamasını al
3. En düşük ortalama test hatasını veren lambda'yı seç

<div style="display: flex; gap: 20px;">
  <img src="../figures/reg-params_lasso.png" width="45%"/>
</div>


Grafiği okuma rehberi:
- Mavi çizgi = eğitim hatası: lambda arttıkça sürekli artar
- Kırmızı çizgi = validation hatası: U şeklinde, önce düşer sonra artar
- Yeşil nokta/çizgi = optimal lambda, validation hatasının en düşük olduğu yer
- Sol bölge = overfitting (lambda çok küçük)
- Sağ bölge = underfitting (lambda çok büyük)

<div style="display: flex; gap: 20px;">
  <img src="../figures/reg-strength.png" width="45%"/>
</div>


Bu grafik lambda arttıkça kaç özelliğin modelde kaldığını gösterir.
Optimal lambda noktasında gerçek önemli özellik sayısına yaklaşılır.

#

In [2]:
import numpy as np


class LassoRegression:
    def __init__(self, lambda_=1.0, max_iter=1000, tol=1e-4):
        """
        lambda_  : L1 ceza parametresi
        max_iter : maksimum iterasyon sayısı
        tol      : yakınsama toleransı (katsayılar bu kadar değişmiyorsa dur)
        """
        self.lambda_ = lambda_
        self.max_iter = max_iter
        self.tol = tol
        self.coef_ = None
        self.intercept_ = None
        self._mu = None
        self._sigma = None

    def _standardize_fit(self, X):
        self._mu = X.mean(axis=0)
        self._sigma = X.std(axis=0)
        self._sigma[self._sigma == 0] = 1
        return (X - self._mu) / self._sigma

    def _standardize_transform(self, X):
        return (X - self._mu) / self._sigma

    @staticmethod
    def _soft_threshold(z, lam):
        if z > lam:
            return z - lam
        elif z < -lam:
            return z + lam
        return 0.0

    def fit(self, X, y):
        """
        Coordinate Descent ile LASSO çözümü.
        Her iterasyonda bir katsayıyı optimize eder, diğerlerini sabit tutar.
        """
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        n, p = X.shape

        X_scaled = self._standardize_fit(X)

        # Intercept için merkeze al
        y_mean = y.mean()
        y_centered = y - y_mean

        # Katsayıları sıfırdan başlat
        beta = np.zeros(p)

        for iteration in range(self.max_iter):
            beta_old = beta.copy()

            for j in range(p):
                # j. katsayı hariç tüm özelliklerin katkısını çıkar
                residual = y_centered - X_scaled @ beta + X_scaled[:, j] * beta[j]

                # Partial residual inner product
                rho = X_scaled[:, j] @ residual

                # Soft-thresholding uygula
                beta[j] = self._soft_threshold(rho / n, self.lambda_) / (
                    (X_scaled[:, j] ** 2).sum() / n
                )

            # Yakınsama kontrolü
            if np.max(np.abs(beta - beta_old)) < self.tol:
                break

        self.coef_ = beta
        self.intercept_ = y_mean  # standardize sonrası intercept = y_mean

        return self

    def predict(self, X):
        X = np.array(X, dtype=float)
        X_scaled = self._standardize_transform(X)
        return X_scaled @ self.coef_ + self.intercept_

    def score(self, X, y):
        y = np.array(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1 - ss_res / ss_tot

    def mse(self, X, y):
        y = np.array(y, dtype=float)
        return np.mean((y - self.predict(X)) ** 2)

    def __repr__(self):
        return f"LassoRegression(lambda_={self.lambda_})"


# --- Test ---
if __name__ == "__main__":
    np.random.seed(42)

    n, p = 100, 5
    X = np.random.randn(n, p)

    # Sadece ilk 2 özellik gerçekten önemli
    true_beta = np.array([3.0, -2.0, 0.0, 0.0, 0.0])
    y = X @ true_beta + np.random.randn(n) * 0.5

    split = int(0.8 * n)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    print("Gerçek beta  :", true_beta)
    print()

    lambdas = [0.01, 0.05, 0.1, 0.5, 1.0]
    print(f"{'Lambda':>8}  {'R²':>8}  {'MSE':>10}  {'Sıfır olan katsayı':>20}")
    print("-" * 55)
    for lam in lambdas:
        m = LassoRegression(lambda_=lam)
        m.fit(X_train, y_train)
        zeros = np.sum(m.coef_ == 0)
        print(f"{lam:>8.2f}  {m.score(X_test, y_test):>8.4f}  "
              f"{m.mse(X_test, y_test):>10.4f}  {zeros:>20}")

    print()
    best = LassoRegression(lambda_=0.05)
    best.fit(X_train, y_train)
    print("Tahmin beta  :", np.round(best.coef_, 4))
    print("Gerçek beta  :", true_beta)

Gerçek beta  : [ 3. -2.  0.  0.  0.]

  Lambda        R²         MSE    Sıfır olan katsayı
-------------------------------------------------------
    0.01    0.9859      0.1658                     2
    0.05    0.9831      0.1978                     3
    0.10    0.9807      0.2256                     3
    0.50    0.9386      0.7200                     3
    1.00    0.8278      2.0176                     3

Tahmin beta  : [ 2.597  -1.7041  0.      0.      0.    ]
Gerçek beta  : [ 3. -2.  0.  0.  0.]


#

## LASSO'nun Avantajları ve Dezavantajları

| | Açıklama |
|---|---|
| **Avantaj** | Otomatik özellik seçimi yapar — gereksiz katsayıları tam sıfır yapar |
| **Avantaj** | Yüksek boyutlu veride (p > n) etkilidir |
| **Avantaj** | Model yorumlanabilirliği artar — az özellik, net ilişki |
| **Avantaj** | Overfitting'i önler |
| **Dezavantaj** | Kapalı form çözümü yoktur, iterasyon gerekir |
| **Dezavantaj** | Yüksek korelasyonlu özelliklerden birini rastgele seçer, diğerini sıfırlar |
| **Dezavantaj** | Tüm özellikler gerçekten önemliyse Ridge daha iyi performans verir |
| **Dezavantaj** | $\lambda$ seçimi kritiktir, cross-validation zorunludur |

## Ridge vs LASSO vs Elastic Net: Özet

<div style="display: flex; gap: 20px;">
  <img src="../figures/coeff-paths.png" width="45%"/>
</div>

| | OLS | Ridge (L2) | LASSO (L1) | Elastic Net |
|---|---|---|---|---|
| Ceza | Yok | $\lambda\sum\beta_j^2$ | $\lambda\sum|\beta_j|$ | İkisi birden |
| Katsayı sıfır olur mu? | — | Hayır | Evet | Evet |
| Özellik seçimi | Hayır | Hayır | Evet | Kısmen |
| Kapalı form | Evet | Evet | Hayır | Hayır |
| Korelasyonlu özellikler | Zayıf | Güçlü | Kararsız | Güçlü |
| Ne zaman kullanılır | Baseline | Tüm özellikler önemliyse | Gereksiz özellik çoksa | Her ikisi gerekiyorsa |